# Skråfoto 3D Reconstruction

Fetch oblique aerial imagery from the Danish Skråfoto API and reconstruct an interactive 3D point cloud.

**Run all cells** (`Shift+Enter` through each), then use the form at the bottom to start a reconstruction.

In [1]:
## ── 1. Environment setup ──────────────────────────────────────────────────
import os, sys
from pathlib import Path

# Make the project root importable
PROJECT_ROOT = Path("__file__").parent.resolve() if "__file__" in dir() else Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Auto-load token from .env if present
_env = PROJECT_ROOT / ".env"
if _env.exists():
    for _line in _env.read_text().splitlines():
        _line = _line.strip()
        if _line and not _line.startswith("#") and "=" in _line:
            _k, _v = _line.split("=", 1)
            os.environ.setdefault(_k.strip(), _v.strip())

# Ensure plotly and ipywidgets are available
try:
    import plotly, ipywidgets
except ImportError:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "plotly", "ipywidgets"], check=True)

print("✓ Environment ready")

✓ Environment ready


In [2]:
## ── 2. Project imports ────────────────────────────────────────────────────
import logging, traceback
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output
import plotly.graph_objects as go
import open3d as o3d

import config
from api.client import SkraafotoClient
from storage.session import Session
from reconstruction.camera import build_cameras
from reconstruction.pipeline import run as run_pipeline

# Quiet the root logger; notebook output widget captures INFO
logging.basicConfig(level=logging.INFO, format="%(levelname)s  %(name)s: %(message)s", force=True)

print("✓ Imports OK")

✓ Imports OK


In [3]:
## ── 3. 3-D viewer ─────────────────────────────────────────────────────────

def view_pointcloud(ply_path: str, max_points: int = 60_000,
                    clip_percentile: float = 1.0) -> go.Figure:
    """
    Load a PLY and return an interactive Plotly 3-D scatter figure.

    Parameters
    ----------
    max_points       : subsample to this many points for browser performance.
    clip_percentile  : remove the outermost N% of points on each axis (removes
                       stray triangulation artifacts while keeping the scene).
    """
    pcd = o3d.io.read_point_cloud(str(ply_path))
    n_total = len(pcd.points)
    if n_total == 0:
        raise ValueError(f"PLY file has no points: {ply_path}")

    pts_all = np.asarray(pcd.points)
    clr_all = np.asarray(pcd.colors) if pcd.has_colors() else None

    # ── Clip outliers by percentile on each axis ──────────
    lo, hi = clip_percentile, 100 - clip_percentile
    mask = np.ones(n_total, dtype=bool)
    for ax in range(3):
        lo_v, hi_v = np.percentile(pts_all[:, ax], [lo, hi])
        mask &= (pts_all[:, ax] >= lo_v) & (pts_all[:, ax] <= hi_v)

    pts_clip = pts_all[mask]
    clr_clip = clr_all[mask] if clr_all is not None else None
    n_clip = len(pts_clip)

    # ── Subsample ─────────────────────────────────────────
    if n_clip > max_points:
        idx = np.random.choice(n_clip, max_points, replace=False)
        pts = pts_clip[idx]
        clr = clr_clip[idx] if clr_clip is not None else None
    else:
        pts = pts_clip
        clr = clr_clip

    # ── Colour ────────────────────────────────────────────
    if clr is not None:
        clr_int = (np.clip(clr, 0, 1) * 255).astype(np.uint8)
        marker_color = [f"rgb({r},{g},{b})" for r, g, b in clr_int]
        marker = dict(size=1.5, color=marker_color, opacity=0.85)
    else:
        marker = dict(size=1.5, color=pts[:, 2], colorscale="Viridis", opacity=0.85)

    # ── Figure ────────────────────────────────────────────
    fig = go.Figure(
        go.Scatter3d(
            x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
            mode="markers",
            marker=marker,
            hoverinfo="skip",
        )
    )
    title_txt = (
        f"Skråfoto 3D — {Path(str(ply_path)).name}  "
        f"({len(pts):,} pts shown / {n_total:,} total)"
    )
    fig.update_layout(
        title=dict(text=title_txt, x=0.5),
        scene=dict(
            xaxis_title="East (m)",
            yaxis_title="North (m)",
            zaxis_title="Up (m)",
            aspectmode="data",
            bgcolor="rgb(15,15,20)",
            xaxis=dict(showbackground=False, color="white"),
            yaxis=dict(showbackground=False, color="white"),
            zaxis=dict(showbackground=False, color="white"),
        ),
        paper_bgcolor="rgb(20,20,25)",
        font=dict(color="white"),
        margin=dict(l=0, r=0, t=40, b=0),
        height=650,
    )
    return fig



def view_pointcloud_pairs(pair_stats: list, pair_view_xyz: list, pair_view_rgb: list) -> "go.Figure":
    """
    Build an interactive 3-D scatter figure with one trace per stereo pair.
    Click a legend entry to toggle that pair on/off.
    """
    # 24-colour qualitative palette (Plotly Dark24)
    _PALETTE = [
        "#2E91E5","#E15F99","#1CA71C","#FB0D0D","#DA16FF","#B68100",
        "#EB663B","#511CFB","#00A08B","#FB00D1","#FC0080","#B2828D",
        "#6C7C32","#778AAE","#862A16","#A777F1","#620042","#1616A7",
        "#DA60CA","#6C4516","#0D2A63","#AF0038","#FD3216","#00FE35",
    ]

    traces = []
    for i, (ps, xyz, rgb) in enumerate(zip(pair_stats, pair_view_xyz, pair_view_rgb)):
        label = (
            f'#{ps["idx"]:02d} {ps["dir_i"]}↔{ps["dir_j"]}  '
            f'({ps["points"]:,} pts)'
        )
        color = _PALETTE[i % len(_PALETTE)]

        if not ps["ok"] or xyz is None:
            # Add an invisible placeholder so the legend entry still shows (greyed out)
            traces.append(go.Scatter3d(
                x=[], y=[], z=[],
                mode="markers",
                name=f'#{ps["idx"]:02d} {ps["dir_i"]}↔{ps["dir_j"]}  FAILED',
                marker=dict(size=2, color=color),
                visible=True,
            ))
            continue

        clr_int = (np.clip(rgb, 0, 1) * 255).astype("uint8")
        marker_color = [f"rgb({r},{g},{b})" for r, g, b in clr_int]

        traces.append(go.Scatter3d(
            x=xyz[:, 0], y=xyz[:, 1], z=xyz[:, 2],
            mode="markers",
            name=label,
            marker=dict(size=1.5, color=marker_color, opacity=0.80),
            hoverinfo="skip",
        ))

    fig = go.Figure(data=traces)
    fig.update_layout(
        title=dict(text="Skråfoto 3D — per-pair view  (click legend to toggle)", x=0.5),
        scene=dict(
            xaxis_title="East (m)",
            yaxis_title="North (m)",
            zaxis_title="Up (m)",
            aspectmode="data",
            bgcolor="rgb(15,15,20)",
            xaxis=dict(showbackground=False, color="white"),
            yaxis=dict(showbackground=False, color="white"),
            zaxis=dict(showbackground=False, color="white"),
        ),
        paper_bgcolor="rgb(20,20,25)",
        font=dict(color="white"),
        legend=dict(
            bgcolor="rgba(30,30,35,0.85)",
            bordercolor="#555",
            borderwidth=1,
            font=dict(size=11),
        ),
        margin=dict(l=0, r=0, t=40, b=0),
        height=700,
    )
    return fig

print("✓ Viewer ready")

✓ Viewer ready


In [ ]:
## ── 4. Reconstruction form ────────────────────────────────────────────────

# ── Input mode ──────────────────────────────────────────
mode = widgets.ToggleButtons(
    options=["Address", "Lon / Lat", "Load existing session"],
    description="",
    button_style="",
    style={"button_width": "175px", "description_width": "0"},
)

address_input = widgets.Text(
    placeholder='e.g. "Rådhuspladsen 1, København"',
    layout=widgets.Layout(width="400px"),
)
lonlat_input = widgets.Text(
    placeholder="lon, lat  e.g.  12.5683, 55.6761",
    layout=widgets.Layout(width="300px"),
)

# ── Session folder selector ──────────────────────────────
DATA_DIR = Path("./data")

def _list_sessions():
    if not DATA_DIR.exists():
        return []
    folders = sorted(
        [d for d in DATA_DIR.iterdir() if d.is_dir()],
        key=lambda d: d.stat().st_mtime,
        reverse=True,
    )
    return [(d.name, str(d)) for d in folders]

_initial_sessions = _list_sessions()
_initial_options   = [label for label, _ in _initial_sessions] or ["(no sessions found)"]
_initial_paths     = {label: path for label, path in _initial_sessions}

session_dropdown = widgets.Dropdown(
    options=_initial_options,
    description="Session:",
    style={"description_width": "70px"},
    layout=widgets.Layout(width="480px"),
    disabled=not bool(_initial_sessions),
)

refresh_btn = widgets.Button(
    description="↺ Refresh",
    button_style="",
    layout=widgets.Layout(width="90px", height="32px"),
    tooltip="Rescan the data/ folder for sessions",
)

def _refresh_sessions(btn=None):
    sessions = _list_sessions()
    if sessions:
        labels = [label for label, _ in sessions]
        _initial_paths.clear()
        _initial_paths.update({label: path for label, path in sessions})
        session_dropdown.options  = labels
        session_dropdown.disabled = False
    else:
        session_dropdown.options  = ["(no sessions found)"]
        session_dropdown.disabled = True

refresh_btn.on_click(_refresh_sessions)

session_box = widgets.VBox([
    widgets.Label("Choose an existing session folder:"),
    widgets.HBox([session_dropdown, refresh_btn]),
])

address_box = widgets.HBox([widgets.Label("Address:"), address_input])
lonlat_box  = widgets.HBox([widgets.Label("Lon, Lat:"), lonlat_input])

def _update_inputs(change):
    address_box.layout.display = "flex"  if mode.value == "Address"               else "none"
    lonlat_box.layout.display  = "flex"  if mode.value == "Lon / Lat"             else "none"
    session_box.layout.display = "flex"  if mode.value == "Load existing session" else "none"
    if mode.value == "Load existing session":
        _refresh_sessions()
mode.observe(_update_inputs, names="value")
_update_inputs(None)

# ── Model settings (always visible) ──────────────────────
scene_radius_w = widgets.IntSlider(
    value=int(config.SCENE_RADIUS), min=100, max=2000, step=50,
    description="Area radius (m):",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="420px"),
    tooltip="XY crop radius around the target location after reconstruction",
)
max_points_w = widgets.IntSlider(
    value=100_000, min=10_000, max=500_000, step=10_000,
    description="Max pts / pair:",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="420px"),
    tooltip="Maximum points sampled per stereo pair — higher = denser but slower",
)

model_settings = widgets.VBox([
    widgets.HTML("<b>Model settings</b>"),
    scene_radius_w,
    max_points_w,
], layout=widgets.Layout(
    border="1px solid #555",
    padding="8px 12px",
    margin="4px 0",
    border_radius="4px",
))

# ── Advanced settings ────────────────────────────────────
collection_w = widgets.Dropdown(
    options=["skraafotos2023", "skraafotos2021", "skraafotos2019"],
    value="skraafotos2023",
    description="Collection:",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="270px"),
)
max_images_w = widgets.IntSlider(value=30, min=5, max=100, step=1,
    description="Max images:", style={"description_width": "90px"},
    layout=widgets.Layout(width="350px"))
max_pairs_w = widgets.IntSlider(value=20, min=1, max=100, step=1,
    description="Max pairs:", style={"description_width": "90px"},
    layout=widgets.Layout(width="350px"))
save_path_w = widgets.Text(value="out.ply", description="Save PLY:",
    style={"description_width": "90px"}, layout=widgets.Layout(width="270px"))

advanced = widgets.Accordion(children=[
    widgets.VBox([
        widgets.HBox([collection_w, save_path_w]),
        max_images_w,
        max_pairs_w,
    ])
])
advanced.set_title(0, "⚙ Advanced settings")
advanced.selected_index = None

# ── Progress bars ─────────────────────────────────────────
def _make_step(label, color="#4c8cbf"):
    bar = widgets.IntProgress(
        min=0, max=1, value=0,
        layout=widgets.Layout(width="260px", height="16px"),
        style={"bar_color": color},
    )
    lbl = widgets.Label("–", layout=widgets.Layout(min_width="220px"))
    return bar, lbl, widgets.HBox([
        widgets.Label(label, layout=widgets.Layout(width="110px")),
        bar, lbl,
    ])

_bar_pairs,  _lbl_pairs,  _row_pairs  = _make_step("Pair selection", "#5a9e6f")
_bar_stereo, _lbl_stereo, _row_stereo = _make_step("Stereo pairs",   "#4c8cbf")
_bar_merge,  _lbl_merge,  _row_merge  = _make_step("Merge & filter", "#c47e2a")

progress_box = widgets.VBox([
    widgets.HTML("<b>Progress</b>"),
    _row_pairs,
    _row_stereo,
    _row_merge,
], layout=widgets.Layout(
    border="1px solid #555",
    padding="8px 12px",
    margin="4px 0",
    border_radius="4px",
    display="none",
))

def _reset_progress():
    for bar, lbl in [(_bar_pairs, _lbl_pairs), (_bar_stereo, _lbl_stereo), (_bar_merge, _lbl_merge)]:
        bar.max = 1; bar.value = 0; lbl.value = "–"
    progress_box.layout.display = "flex"

def _flush():
    """Force pending ipywidgets comm messages to reach the browser."""
    import time
    try:
        import IPython
        ip = IPython.get_ipython()
        if ip and hasattr(ip, 'kernel'):
            ip.kernel._publish_status('busy', parent=ip.kernel._parent_header)
    except Exception:
        pass
    time.sleep(0.05)

def _on_progress(step, message, done, total):
    if step == "pairs":
        _bar_pairs.max = max(total, 1); _bar_pairs.value = done; _lbl_pairs.value = message
        _bar_pairs.send_state(); _lbl_pairs.send_state()
    elif step == "stereo":
        _bar_stereo.max = max(total, 1); _bar_stereo.value = done; _lbl_stereo.value = message
        _bar_stereo.send_state(); _lbl_stereo.send_state()
    elif step == "merge":
        _bar_merge.max = max(total, 1); _bar_merge.value = done; _lbl_merge.value = message
        _bar_merge.send_state(); _lbl_merge.send_state()
    _flush()

# ── Run button + log output + viewer output ──────────────
run_btn    = widgets.Button(description="▶  Run Reconstruction",
                             button_style="success",
                             layout=widgets.Layout(width="220px", height="38px"))
log_out    = widgets.Output(layout=widgets.Layout(
                border="1px solid #444", max_height="280px", overflow_y="auto",
                padding="6px", margin="6px 0"))
viewer_out = widgets.Output()

# ── Full form layout ─────────────────────────────────────
form = widgets.VBox([
    widgets.HTML("<h3 style='margin:8px 0'>Skråfoto 3D Reconstruction</h3>"),
    mode,
    address_box, lonlat_box, session_box,
    model_settings,
    advanced,
    run_btn,
    progress_box,
    log_out,
    viewer_out,
], layout=widgets.Layout(padding="12px", max_width="680px"))

# ── UTM helpers ──────────────────────────────────────────
def _lonlat_to_utm32n(lon, lat):
    import math
    a, f = 6378137.0, 1 / 298.257223563
    e2 = 2 * f - f ** 2
    k0, lon0, E0 = 0.9996, math.radians(9.0), 500_000.0
    phi, lam = math.radians(lat), math.radians(lon) - lon0
    N = a / math.sqrt(1 - e2 * math.sin(phi) ** 2)
    T, C = math.tan(phi) ** 2, e2 / (1 - e2) * math.cos(phi) ** 2
    A = math.cos(phi) * lam
    M = a * (
        (1 - e2/4 - 3*e2**2/64) * phi
        - (3*e2/8 + 3*e2**2/32) * math.sin(2*phi)
        + (15*e2**2/256) * math.sin(4*phi)
    )
    E = k0 * N * (A + (1-T+C)*A**3/6) + E0
    N_utm = k0 * (M + N*math.tan(phi)*(A**2/2 + (5-T+9*C)*A**4/24))
    return E, N_utm

def _scene_center_from_session(session_path, origin_utm):
    import json
    sess_json = Path(session_path) / "session.json"
    if not sess_json.exists():
        return None
    data = json.loads(sess_json.read_text())
    lonlat = data.get("query", {}).get("lonlat")
    if lonlat is None:
        return None
    lon, lat = lonlat
    e, n = _lonlat_to_utm32n(lon, lat)
    return np.array([e - origin_utm[0], n - origin_utm[1], 0.0])

def _estimate_ground_z(origin_utm):
    """Estimate the ground plane Z in local ENU from the mean camera altitude.
    Assumes ground ≈ 20 m above sea level (Denmark); cameras are at origin_utm[2]."""
    return 20.0 - float(origin_utm[2])

# ── Run state guard (prevents multiple simultaneous runs) ─
_run_state = {"active": False}

# ── Run handler ──────────────────────────────────────────
def _on_run(btn):
    if _run_state["active"]:
        return
    _run_state["active"] = True
    log_out.clear_output()
    viewer_out.clear_output()
    _reset_progress()
    run_btn.disabled = True
    run_btn.description = "⏳  Running…"
    run_btn.send_state()

    try:
        _run_reconstruction()
    except Exception as exc:
        with log_out:
            print(f"\n❌  Error: {exc}")
            traceback.print_exc()
    finally:
        _run_state["active"] = False
        run_btn.disabled = False
        run_btn.description = "▶  Run Reconstruction"

def _run_reconstruction():
    save_path    = save_path_w.value.strip() or "out.ply"
    collection   = collection_w.value
    max_images   = max_images_w.value
    max_pairs    = max_pairs_w.value
    scene_radius = scene_radius_w.value
    max_points   = max_points_w.value

    # ── Resolve location ─────────────────────────────────
    if mode.value == "Load existing session":
        label = session_dropdown.value
        if label == "(no sessions found)" or session_dropdown.disabled:
            raise ValueError("No sessions found. Run a reconstruction first to create one.")
        folder = _initial_paths.get(label, label)
        with log_out:
            print(f"Loading session: {folder}")
        items, images = Session.load_from_path(folder)
        session_folder = Path(folder)

        cameras, origin_utm = build_cameras(items, images, session_folder=session_folder)
        scene_center = _scene_center_from_session(folder, origin_utm)

    else:
        token = config.get_token()
        client = SkraafotoClient(token)

        if mode.value == "Address":
            addr = address_input.value.strip()
            if not addr:
                raise ValueError("Enter a Danish address.")
            with log_out:
                print(f"Geocoding: {addr}")
            lon, lat = client.geocode(addr)
            location_label = addr
        else:
            raw = lonlat_input.value.strip()
            if not raw:
                raise ValueError("Enter lon, lat coordinates.")
            parts = raw.split(",")
            lon, lat = float(parts[0].strip()), float(parts[1].strip())
            location_label = f"lon{lon:.4f}_lat{lat:.4f}"

        with log_out:
            print(f"Location: {location_label}  ({lon:.6f}, {lat:.6f})")

        import math
        r_m = 200.0
        bbox = [
            lon - r_m / (111320 * math.cos(math.radians(lat))),
            lat - r_m / 111320,
            lon + r_m / (111320 * math.cos(math.radians(lat))),
            lat + r_m / 111320,
        ]

        with log_out:
            print("Searching API…")
        items = client.search(bbox, collection=collection, max_images=max_images)
        if len(items) < 2:
            raise RuntimeError(f"Only {len(items)} image(s) found. Try a different location or collection.")

        n_items = len(items)
        with log_out:
            print(f"Found {n_items} images. Downloading…")
        _bar_pairs.max = n_items; _bar_pairs.value = 0
        _lbl_pairs.value = f"0 / {n_items}"

        import cv2 as _cv2
        session = Session(base_dir="./data", lon=lon, lat=lat,
                          address=addr if mode.value == "Address" else None)

        images = {}
        for k, item in enumerate(items):
            session.save_item_metadata(item)
            img_path = session.image_path(item.id)
            if not img_path.exists():
                try:
                    raw_bytes = client.download_image_bytes(item.full_href)
                    session.save_image_bytes(item.id, raw_bytes)
                except Exception as exc:
                    with log_out:
                        print(f"  ⚠ Download failed {item.id}: {exc}")
                    continue
            arr = np.frombuffer(img_path.read_bytes(), dtype=np.uint8)
            img = _cv2.imdecode(arr, _cv2.IMREAD_COLOR)
            if img is not None:
                images[item.id] = img
            _bar_pairs.value = k + 1
            _lbl_pairs.value = f"{k + 1} / {n_items}"
            _bar_pairs.send_state(); _lbl_pairs.send_state()

        session.write_session_json(
            {"lonlat": [lon, lat], "collection": collection},
            [it for it in items if it.id in images],
        )
        session_folder = session.folder
        with log_out:
            print(f"Session saved to: {session_folder}")
        _refresh_sessions()

        cameras, origin_utm = build_cameras(items, images, session_folder=session_folder)

        target_e, target_n = _lonlat_to_utm32n(lon, lat)
        scene_center = np.array([target_e - origin_utm[0], target_n - origin_utm[1], 0.0])

    ground_z = _estimate_ground_z(origin_utm)
    if scene_center is not None:
        with log_out:
            print(f"Scene centre (ENU): ({scene_center[0]:.1f}, {scene_center[1]:.1f}) m  "
                  f"ground_z≈{ground_z:.0f} m")

    # ── Run stereo pipeline ───────────────────────────────
    with log_out:
        print(f"Running stereo pipeline ({max_pairs} pairs, radius={scene_radius} m, max_pts={max_points:,})…")
    pcd, stats = run_pipeline(
        cameras,
        max_pairs=max_pairs,
        min_baseline=config.MIN_BASELINE,
        max_baseline=config.MAX_BASELINE,
        debug=False,
        session_folder=session_folder,
        scene_center=scene_center,
        scene_radius=scene_radius,
        max_points=max_points,
        progress_callback=_on_progress,
        ground_z=ground_z,
    )

    # ── Save PLY ──────────────────────────────────────────
    o3d.io.write_point_cloud(save_path, pcd)

    with log_out:
        print("")
        print("═" * 45)
        print(f"Session:        {session_folder}")
        print(f"Pairs:          {stats['pairs_processed']} / {stats['pairs_selected']}")
        print(f"Points (raw):   {stats['points_before_merge']:,}")
        print(f"Points (final): {stats['points_after_merge']:,}")
        print(f"PLY saved:      {save_path}")
        print("═" * 45)
        print("")
        # ── Per-pair stats table ─────────────────────────
        ps = stats.get("pair_stats", [])
        if ps:
            print(f"  {"#":>3}  {"Pair":<30}  {"Baseline":>10}  {"Score":>7}  {"Points":>9}  Status")
            print("  " + "-" * 72)
            for p in ps:
                pair_label = f'{p["dir_i"]}/{p["id_i"]} ↔ {p["dir_j"]}/{p["id_j"]}'
                status = "ok" if p["ok"] else "FAILED"
                pts_str = f'{p["points"]:,}' if p["ok"] else "—"
                print(f'  {p["idx"]:>3}  {pair_label:<30}  {p["baseline"]:>9.0f}m  {p["score"]:>7.3f}  {pts_str:>9}  {status}')

    # ── Show 3-D viewer (per-pair) ───────────────────────
    with viewer_out:
        print("Rendering per-pair 3D viewer…")
        clear_output(wait=True)
        fig = view_pointcloud_pairs(
            stats.get("pair_stats", []),
            stats.get("pair_view_xyz", []),
            stats.get("pair_view_rgb", []),
        )
        fig.show()

# Guard against duplicate handlers when cell is re-run without kernel restart
run_btn._click_handlers.callbacks.clear()
run_btn.on_click(_on_run)
display(form)


---
### Quick viewer — open any PLY directly

Run the cell below to load an existing PLY file without re-running reconstruction.

In [5]:
view_pointcloud("out.ply").show()